In [1]:
import pandas as pd
import numpy as np
import os
import glob

ModuleNotFoundError: No module named 'pandas'

In [2]:
import pandas as pd
import numpy as np
import os
import glob

In [3]:
DATA_PATH = "../data/raw"

files = glob.glob(os.path.join(DATA_PATH, "*.csv"))

print(f"Found {len(files)} CSV files:\n")

for file in files:
    print(os.path.basename(file))

Found 8 CSV files:

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv


In [4]:
for file in files:
    size_mb = os.path.getsize(file) / (1024 ** 2)
    print(f"{os.path.basename(file):65} {size_mb:8.1f} MB")

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv           79.3 MB
Monday-WorkingHours.pcap_ISCX.csv                                    168.7 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv                             55.6 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv                  73.3 MB
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                      73.6 MB
Tuesday-WorkingHours.pcap_ISCX.csv                                   128.8 MB
Wednesday-workingHours.pcap_ISCX.csv                                 214.7 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv                49.6 MB


In [5]:
monday_file = [
    f for f in files
    if "Monday" in os.path.basename(f)
][0]

print(monday_file)

../data/raw/Monday-WorkingHours.pcap_ISCX.csv


In [6]:
df_monday = pd.read_csv(monday_file)

In [7]:
print("Shape:", df_monday.shape)

Shape: (529918, 79)


In [8]:
df_monday.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
df_monday.info()

<class 'pandas.DataFrame'>
RangeIndex: 529918 entries, 0 to 529917
Data columns (total 79 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0    Destination Port             529918 non-null  int64  
 1    Flow Duration                529918 non-null  int64  
 2    Total Fwd Packets            529918 non-null  int64  
 3    Total Backward Packets       529918 non-null  int64  
 4   Total Length of Fwd Packets   529918 non-null  int64  
 5    Total Length of Bwd Packets  529918 non-null  int64  
 6    Fwd Packet Length Max        529918 non-null  int64  
 7    Fwd Packet Length Min        529918 non-null  int64  
 8    Fwd Packet Length Mean       529918 non-null  float64
 9    Fwd Packet Length Std        529918 non-null  float64
 10  Bwd Packet Length Max         529918 non-null  int64  
 11   Bwd Packet Length Min        529918 non-null  int64  
 12   Bwd Packet Length Mean       529918 non-null  float64


In [10]:
labels = {}

for file in files:
    temp = pd.read_csv(file, usecols=lambda column: "Label" in column)
    
    temp.columns = temp.columns.str.strip()
    
    label_column = temp.columns[0]
    
    labels[os.path.basename(file)] = temp[label_column].value_counts()

In [11]:
for filename, counts in labels.items():
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)
    print(counts)



Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
Label
BENIGN    529918
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Tuesday-WorkingHours.pcap_ISCX.csv
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Wednesday-workingHours.pcap_ISCX.csv
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Label


In [12]:
all_labels = pd.concat(labels.values(), axis=1).fillna(0)

all_labels["Total"] = all_labels.sum(axis=1)

all_labels = all_labels.sort_values("Total", ascending=False)

all_labels

,count,count,count,count,count,count,count,count,Total
Label,,,,,,,,,
BENIGN,288566.0,529918.0,189067.0,127537.0,97718.0,432074.0,440031.0,168186.0,2273097.0
DoS Hulk,0.0,0.0,0.0,0.0,0.0,0.0,231073.0,0.0,231073.0
PortScan,0.0,0.0,0.0,158930.0,0.0,0.0,0.0,0.0,158930.0
DDoS,0.0,0.0,0.0,0.0,128027.0,0.0,0.0,0.0,128027.0
DoS GoldenEye,0.0,0.0,0.0,0.0,0.0,0.0,10293.0,0.0,10293.0
FTP-Patator,0.0,0.0,0.0,0.0,0.0,7938.0,0.0,0.0,7938.0
SSH-Patator,0.0,0.0,0.0,0.0,0.0,5897.0,0.0,0.0,5897.0
DoS slowloris,0.0,0.0,0.0,0.0,0.0,0.0,5796.0,0.0,5796.0
DoS Slowhttptest,0.0,0.0,0.0,0.0,0.0,0.0,5499.0,0.0,5499.0


In [13]:
missing = df_monday.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

Flow Bytes/s    64
dtype: int64

In [14]:
print("Total missing values:", df_monday.isnull().sum().sum())

Total missing values: 64


In [15]:
print("Duplicate rows:", df_monday.duplicated().sum())

Duplicate rows: 26935


In [16]:
numeric_columns = df_monday.select_dtypes(include=np.number).columns

infinite_count = np.isinf(
    df_monday[numeric_columns]
).sum().sum()

print("Infinite values:", infinite_count)

Infinite values: 810


In [17]:
df_monday.describe().T

/Users/nishkamehta/Documents/ml-intrusion-detection/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/nishkamehta/Documents/ml-intrusion-detection/.venv/lib/python3.11/site-packages/pandas/core/nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,count,mean,std,min,25%,50%,75%,max
Destination Port,529918.0,1.064437e+04,2.139021e+04,0.0,53.0,80.0,443.00,6.553500e+04
Flow Duration,529918.0,1.038927e+07,2.875195e+07,-1.0,176.0,31303.0,355744.75,1.200000e+08
Total Fwd Packets,529918.0,1.039032e+01,8.924128e+02,1.0,2.0,2.0,4.00,2.197590e+05
Total Backward Packets,529918.0,1.151710e+01,1.173319e+03,0.0,1.0,2.0,3.00,2.919220e+05
Total Length of Fwd Packets,529918.0,5.324195e+02,6.228642e+03,0.0,18.0,68.0,187.00,1.323378e+06
...,...,...,...,...,...,...,...,...
Active Min,529918.0,4.380369e+04,4.993677e+05,0.0,0.0,0.0,0.00,1.016597e+08
Idle Mean,529918.0,3.463918e+06,1.297057e+07,0.0,0.0,0.0,0.00,1.199997e+08
Idle Std,529918.0,2.024408e+05,2.170149e+06,0.0,0.0,0.0,0.00,7.514502e+07
Idle Max,529918.0,3.620657e+06,1.340649e+07,0.0,0.0,0.0,0.00,1.199997e+08


In [18]:
df_monday.shape

(529918, 79)

In [19]:
df_monday.columns.tolist()

[' Destination Port',
 ' Flow Duration',
 ' Total Fwd Packets',
 ' Total Backward Packets',
 'Total Length of Fwd Packets',
 ' Total Length of Bwd Packets',
 ' Fwd Packet Length Max',
 ' Fwd Packet Length Min',
 ' Fwd Packet Length Mean',
 ' Fwd Packet Length Std',
 'Bwd Packet Length Max',
 ' Bwd Packet Length Min',
 ' Bwd Packet Length Mean',
 ' Bwd Packet Length Std',
 'Flow Bytes/s',
 ' Flow Packets/s',
 ' Flow IAT Mean',
 ' Flow IAT Std',
 ' Flow IAT Max',
 ' Flow IAT Min',
 'Fwd IAT Total',
 ' Fwd IAT Mean',
 ' Fwd IAT Std',
 ' Fwd IAT Max',
 ' Fwd IAT Min',
 'Bwd IAT Total',
 ' Bwd IAT Mean',
 ' Bwd IAT Std',
 ' Bwd IAT Max',
 ' Bwd IAT Min',
 'Fwd PSH Flags',
 ' Bwd PSH Flags',
 ' Fwd URG Flags',
 ' Bwd URG Flags',
 ' Fwd Header Length',
 ' Bwd Header Length',
 'Fwd Packets/s',
 ' Bwd Packets/s',
 ' Min Packet Length',
 ' Max Packet Length',
 ' Packet Length Mean',
 ' Packet Length Std',
 ' Packet Length Variance',
 'FIN Flag Count',
 ' SYN Flag Count',
 ' RST Flag Count',
 ' P

In [20]:
all_labels

,count,count,count,count,count,count,count,count,Total
Label,,,,,,,,,
BENIGN,288566.0,529918.0,189067.0,127537.0,97718.0,432074.0,440031.0,168186.0,2273097.0
DoS Hulk,0.0,0.0,0.0,0.0,0.0,0.0,231073.0,0.0,231073.0
PortScan,0.0,0.0,0.0,158930.0,0.0,0.0,0.0,0.0,158930.0
DDoS,0.0,0.0,0.0,0.0,128027.0,0.0,0.0,0.0,128027.0
DoS GoldenEye,0.0,0.0,0.0,0.0,0.0,0.0,10293.0,0.0,10293.0
FTP-Patator,0.0,0.0,0.0,0.0,0.0,7938.0,0.0,0.0,7938.0
SSH-Patator,0.0,0.0,0.0,0.0,0.0,5897.0,0.0,0.0,5897.0
DoS slowloris,0.0,0.0,0.0,0.0,0.0,0.0,5796.0,0.0,5796.0
DoS Slowhttptest,0.0,0.0,0.0,0.0,0.0,0.0,5499.0,0.0,5499.0


In [21]:
print("Missing:", df_monday.isnull().sum().sum())
print("Duplicates:", df_monday.duplicated().sum())
print("Infinite:", infinite_count)

Missing: 64
Duplicates: 26935
Infinite: 810
